# UP2.5 · Pandas III: leer y limpiar datos

Ejemplos de la presentación, en el mismo orden. Este notebook va en la raíz del proyecto, junto a la carpeta `datos/`.

Ejecuta las celdas de una en una, de arriba abajo: cada una usa variables de las anteriores. Cada sección termina con una tabla de referencia.

`pisos_sucio.csv` son los mismos anuncios de `pisos.csv` tal como llegarían de una web: con otro formato, huecos, duplicados, texto mal escrito y valores imposibles.

In [1]:
import sqlite3
from pathlib import Path

import numpy as np
import pandas as pd

ruta: Path = Path("datos") / "pisos_sucio.csv"

## 1. Leer de otras fuentes

In [2]:
mal: pd.DataFrame = pd.read_csv(ruta)
print(mal.shape)
print(mal.columns[0][:40])

(614, 1)
id;fecha;barrio;tipo;metros;habitaciones


In [3]:
pisos: pd.DataFrame = pd.read_csv(ruta, sep=";", decimal=",")
print(pisos.shape)
print(pisos[["barrio", "metros", "ascensor", "precio"]].head(3))

(614, 13)
     barrio  metros ascensor   precio
0   Levante   115.0        S  1.830 €
1   Foietes    35.0       No    570 €
2  Poniente    70.0       Sí  1.040 €


In [4]:
print(pisos.dtypes)

id                int64
fecha               str
barrio              str
tipo                str
metros          float64
habitaciones    float64
banos           float64
planta          float64
ascensor            str
terraza             str
vistas_mar          str
anunciante          str
precio              str
dtype: object


In [5]:
ruta_xlsx: Path = Path("datos") / "barrios.xlsx"
print(pd.ExcelFile(ruta_xlsx).sheet_names)

servicios: pd.DataFrame = pd.read_excel(
    ruta_xlsx, sheet_name="servicios"
)
print(servicios.head(3))

['barrios', 'servicios']
           barrio  centros_salud  colegios  paradas_bus
0  Rincón de Loix              1         2            9
1         Levante              1         1            7
2          Centro              2         3           12


In [6]:
ruta_db: Path = Path("datos") / "inmobiliaria.db"
con: sqlite3.Connection = sqlite3.connect(ruta_db)
consulta: str = (
    "SELECT barrio, tipo, precio FROM anuncios "
    "WHERE precio > 3500"
)
muy_caros: pd.DataFrame = pd.read_sql(consulta, con)
con.close()
print(muy_caros)

    barrio    tipo  precio
0  La Cala  dúplex    3620
1  Levante  dúplex    3930


### Referencia: leer de otras fuentes

| Código | Qué hace |
|---|---|
| `pd.read_csv(ruta, sep=";")` | Columnas separadas por punto y coma, habitual en archivos en español |
| `decimal=","` | Los decimales se escriben con coma: `115,0` |
| `encoding="latin-1"` | Si salen caracteres raros (`Ã³` en lugar de `ó`), el archivo no está en UTF-8 |
| `pd.ExcelFile(ruta).sheet_names` | Los nombres de las hojas de un Excel |
| `pd.read_excel(ruta, sheet_name="hoja")` | Una hoja de un Excel (necesita `uv add openpyxl`) |
| `sqlite3.connect(ruta)` | Abre una base de datos SQLite, que es un único archivo |
| `pd.read_sql(consulta, con)` | El resultado de una consulta SQL, como DataFrame |
| `con.close()` | Cierra la conexión |

## 2. Huecos y duplicados

In [7]:
print(pisos.duplicated().sum())
repetidas: pd.DataFrame = pisos[pisos.duplicated(keep=False)]
print(repetidas[["id", "barrio", "precio"]].head(4))

14
      id    barrio   precio
53  1054  Poniente  1.710 €
54  1054  Poniente  1.710 €
61  1061    Ponent  1.360 €
62  1061    Ponent  1.360 €


In [8]:
pisos = pisos.drop_duplicates()
print(len(pisos), pisos["id"].duplicated().sum())

600 0


In [9]:
huecos: pd.Series = pisos.isna().sum()
print(huecos[huecos > 0])

fecha            8
metros           9
habitaciones    17
banos           13
planta          19
anunciante      22
dtype: int64


In [10]:
print(len(pisos.dropna()))

pisos = pisos.dropna(subset=["metros"])
print(len(pisos))

517
591


In [11]:
pisos["anunciante"] = pisos["anunciante"].fillna("desconocido")
print(pisos["anunciante"].value_counts())

anunciante
agencia        374
particular     195
desconocido     22
Name: count, dtype: int64


### Referencia: huecos y duplicados

| Código | Qué hace |
|---|---|
| `df.duplicated()` | True en cada fila que repite una anterior |
| `df.duplicated(keep=False)` | True en todas las copias, también la primera |
| `df.drop_duplicates()` | Quita las filas repetidas y deja la primera |
| `df["id"].duplicated().sum()` | Cuántos valores repetidos hay en una columna |
| `df.isna().sum()` | Cuántos huecos hay en cada columna |
| `df.dropna()` | Quita las filas con algún hueco, en cualquier columna |
| `df.dropna(subset=["metros"])` | Quita solo las filas sin metros |
| `s.fillna(valor)` | Rellena los huecos con un valor |

**Rellenar o no:** un hueco no siempre hay que rellenarlo. Si no se sabe la planta de un piso, `NaN` dice la verdad; inventar un valor la esconde.

## 3. Texto

In [12]:
nombres: pd.Series = pd.Series([" Centro", "PONIENTE", "la cala "])
print(nombres.str.strip().tolist())
print(nombres.str.lower().tolist())
print(nombres.str.strip().str.title().tolist())
print(nombres.str.contains("o").tolist())

['Centro', 'PONIENTE', 'la cala']
[' centro', 'poniente', 'la cala ']
['Centro', 'Poniente', 'La Cala']
[True, False, False]


In [13]:
print(pisos["barrio"].nunique())
print(pisos["barrio"].value_counts().tail(6))

22
barrio
Levante           7
Casco Viejo       6
casco antiguo     6
Racó de l'Oix     6
Rincon de Loix    5
Llevant           2
Name: count, dtype: int64


In [14]:
pisos["barrio"] = pisos["barrio"].str.strip()
print(pisos["barrio"].nunique())

20


In [15]:
VARIANTES: dict[str, str] = {
    "centro": "Centro",
    "PONIENTE": "Poniente", "Ponent": "Poniente",
    "levante": "Levante", "Llevant": "Levante",
    "Casco Viejo": "Casco Antiguo", "casco antiguo": "Casco Antiguo",
    "rincón de loix": "Rincón de Loix", "Rincon de Loix": "Rincón de Loix",
    "Racó de l'Oix": "Rincón de Loix",
}
pisos["barrio"] = pisos["barrio"].replace(VARIANTES)
print(pisos["barrio"].nunique())

10


In [16]:
print(pisos["precio"].head(3).tolist())
print(pisos["precio"].dtype)

['1.830 €', '570 €', '1.040 €']
str


In [17]:
def a_euros(texto: str) -> int:
    return int(texto.replace(" €", "").replace(".", ""))

print(a_euros("1.830 €"))
print(pisos["precio"].apply(a_euros).head(3).tolist())

1830
[1830, 570, 1040]


In [18]:
solo_cifras: pd.Series = pisos["precio"].str.replace(r"\D", "", regex=True)
print(solo_cifras.head(3).tolist())

pisos["precio"] = solo_cifras.astype(int)
print(pisos["precio"].dtype, pisos["precio"].max())

['1830', '570', '1040']
int64 3930


In [19]:
print(pisos["ascensor"].unique())

<StringArray>
['S', 'No', 'Sí', 'si', 'sí', 'no', 'N']
Length: 7, dtype: str


In [20]:
SI_NO: dict[str, bool] = {
    "sí": True, "Sí": True, "si": True, "S": True,
    "no": False, "No": False, "N": False,
}
SI_O_NO: list[str] = ["ascensor", "terraza", "vistas_mar"]
for columna in SI_O_NO:
    pisos[columna] = pisos[columna].map(SI_NO)

print(pisos["ascensor"].value_counts())
print(pisos[SI_O_NO].isna().sum().sum())

ascensor
True     454
False    137
Name: count, dtype: int64
0


### Referencia: texto

| Código | Qué hace |
|---|---|
| `s.str.strip()` | Quita los espacios del principio y del final |
| `s.str.lower()` · `s.str.upper()` · `s.str.title()` | Minúsculas, mayúsculas, inicial de cada palabra en mayúscula |
| `s.str.contains("texto")` | True si el valor contiene ese texto |
| `s.str.replace(" €", "")` | Sustituye un texto por otro dentro de cada valor |
| `s.str.replace(r"\D", "", regex=True)` | Con una expresión regular: `\D` es cualquier carácter que no sea una cifra |
| `s.replace({"viejo": "nuevo"})` | Sustituye valores completos; los que no están en el diccionario no cambian |
| `s.map({"sí": True, "no": False})` | Traduce cada valor con un diccionario; los que no están pasan a `NaN` |
| `s.apply(funcion)` | Llama a una función de Python con cada valor. Úsala solo si no hay un método que lo haga |

**Expresiones regulares básicas:** `\d` una cifra, `\D` cualquier cosa que no sea cifra, `\s` un espacio, `+` una o más veces, `^` inicio y `$` final del texto.

## 4. Fechas, valores imposibles y guardar

In [21]:
print(pisos["fecha"].head(2).tolist())
pisos["fecha"] = pd.to_datetime(pisos["fecha"], format="%d/%m/%Y")
print(pisos["fecha"].dtype, pisos["fecha"].isna().sum())

['02/01/2026', '04/01/2026']
datetime64[us] 8


In [22]:
pisos["mes"] = pisos["fecha"].dt.month
pisos["dia_semana"] = pisos["fecha"].dt.dayofweek
print(pisos[["fecha", "mes", "dia_semana"]].head(3))

       fecha  mes  dia_semana
0 2026-01-02  1.0         4.0
1 2026-01-04  1.0         6.0
2 2026-01-04  1.0         6.0


In [23]:
numericas: list[str] = ["metros", "habitaciones", "planta"]
print(pisos[numericas].describe().loc[["min", "max"]])

     metros  habitaciones  planta
min     0.0          -1.0     0.0
max  8500.0           5.0   210.0


In [24]:
metros: pd.Series = pisos["metros"]
pisos.loc[(metros < 15) | (metros > 1000), "metros"] = np.nan
pisos.loc[pisos["habitaciones"] < 0, "habitaciones"] = np.nan
pisos.loc[pisos["planta"] > 40, "planta"] = np.nan
print(pisos[numericas].describe().loc[["min", "max"]])

     metros  habitaciones  planta
min    28.0           1.0     0.0
max   159.0           5.0    26.0


In [25]:
q1: float = float(pisos["precio"].quantile(0.25))
q3: float = float(pisos["precio"].quantile(0.75))
limite: float = q3 + 1.5 * (q3 - q1)

pisos["precio_atipico"] = pisos["precio"] > limite
print(pisos["precio_atipico"].sum())

13


In [26]:
ruta_limpio: Path = Path("datos") / "pisos_limpio.csv"
pisos.to_csv(ruta_limpio, index=False)
print(pd.read_csv(ruta_limpio).shape)

(591, 16)


In [27]:
con = sqlite3.connect(Path("datos") / "pisos_limpio.db")
pisos.to_sql("anuncios", con, if_exists="replace", index=False)
print(pd.read_sql("SELECT COUNT(*) AS filas FROM anuncios", con))
con.close()

   filas
0    591


### Referencia: fechas, valores imposibles y guardar

| Código | Qué hace |
|---|---|
| `pd.to_datetime(s, format="%d/%m/%Y")` | Texto día/mes/año a fecha. Los huecos pasan a `NaT` |
| `s.dt.month` · `s.dt.year` · `s.dt.day` | Mes, año y día de cada fecha |
| `s.dt.dayofweek` | Día de la semana: 0 es lunes y 6, domingo |
| `df[cols].describe().loc[["min", "max"]]` | Mínimo y máximo de varias columnas, para ver valores imposibles |
| `df.loc[condición, "col"] = np.nan` | Convierte en hueco los valores imposibles |
| `df.to_csv(ruta, index=False)` | Guarda en CSV sin la columna del índice |
| `df.to_sql("tabla", con, if_exists="replace", index=False)` | Guarda en una tabla de SQLite; `replace` la sustituye si ya existe |

**Imposible o atípico:** 0 m² es imposible y se convierte en hueco. Un ático de 3900 € es atípico pero real: se marca y se conserva.